<a href="https://colab.research.google.com/github/0MSA0/supreme-robot/blob/main/notebooks/benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import time
import tiktoken

!pip install thop

from thop import profile
from torch.utils.flop_counter import FlopCounterMode

# Theoretical max flops per second provided by the GPU manufacturer
flops_per_second = {
    # https://www.techpowerup.com/gpu-specs/h100-pcie-80-gb.c3899
    "H100": {
        torch.float32: 51.22e12,  # 51.22 TFLOPs for FP32 on NVIDIA H100
        torch.float16: 204.9e12,  # 204.9 TFLOPs for FP16 on NVIDIA H100
        torch.bfloat16: 204.9e12
    },
    # https://www.techpowerup.com/gpu-specs/l4.c4091
    "L4": {
        torch.float32: 30.29e12,  # 30.29 TFLOPs for FP32 on NVIDIA L4
        torch.float16: 30.29e12,  # 30.29 TFLOPs for FP16 on NVIDIA L4
        torch.bfloat16: 30.29e12
    },
    # https://www.techpowerup.com/gpu-specs/tesla-t4.c3316
    "T4": {
        torch.float32: 8.1e12,  # 8.1 TFLOPs for FP32 on NVIDIA T4
        torch.float16: 65.13e12,  # 65.13 TFLOPs for FP16 on NVIDIA T4
        torch.bfloat16: 65.13e12
    },
    # https://www.techpowerup.com/gpu-specs/a10g.c3798
    "A10G": {
        torch.float32: 31.52e12,  # 31.52 TFLOPs for FP32 on NVIDIA A10G
        torch.float16: 31.52e12,  # 31.52 TFLOPs for FP16 on NVIDIA A10G
        torch.bfloat16: 31.52e12
    },
    # https://www.techpowerup.com/gpu-specs/a100-pcie-40-gb.c3623
    "A100": {
        torch.float32: 19.49e12,  # 19.49 TFLOPs for FP32 on NVIDIA A100
        torch.float16: 77.97e12,  # 77.97 TFLOPs for FP16 on NVIDIA A100
        torch.bfloat16: 77.97e12
    }
}

tokenizer = tiktoken.get_encoding("gpt2")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_configs = {
    "gpt-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG = {
  "vocab_size": tokenizer.n_vocab,  # Vocabulary size
  "context_length": 512,  # Context length
  "temperature": 1.0,  # Temperature
  "drop_rate": 0.1,  # Dropout rate
  "qkv_bias": False,  # Query-Key-Value bias
}

def get_gpu_model(flops_per_second_dict):
    device_name = torch.cuda.get_device_name(0)
    for model in flops_per_second_dict.keys():
        if model in device_name:
            return model
    return "Unknown"  # Default if no matching model is found


gpu_model = get_gpu_model(flops_per_second)
print("GPU Model:", gpu_model)

**Inital model**

In [ ]:
from supreme_robot import SupremeGPTModel

if gpu_model != "Unknown":

    for size in model_configs:
        print(f"\nProcessing {size}")
        config = BASE_CONFIG.copy()
        config.update(model_configs[size])

        min_batch_size = 1
        max_batch_size = None
        max_possible_batch_size = 4096

        while min_batch_size <= max_possible_batch_size:
            batch_size = (min_batch_size + max_possible_batch_size) // 2
            try:
                input_tensor = torch.randint(
                    0, config["vocab_size"],
                    (batch_size, config["context_length"]),
                    device=device
                )

                model = SupremeGPTModel(config).to(device)
                model.train()

                # Start timing
                torch.cuda.synchronize()
                start_time = time.time()

                # Forward & backward pass
                output = model(input_tensor)
                loss = output.sum()  # Compute a dummy loss
                loss.backward()

                # End timing
                torch.cuda.synchronize()
                end_time = time.time()

                total_time_seconds = end_time - start_time

                # Calculate FLOPs for forward pass
                macs, params = profile(model, inputs=(input_tensor,), verbose=False)
                flops_forward = 2 * macs  # Assuming one MAC equals two FLOPs

                # Estimate FLOPs for backward pass (typically 2x forward FLOPs)
                flops_backward = 2 * flops_forward

                # Total FLOPs for forward + backward passes
                total_flops = flops_forward + flops_backward  # Or total_flops = flops_forward * 3

                data_type = next(model.parameters()).dtype
                max_flops_per_second = flops_per_second[gpu_model].get(data_type, 0)

                # Compute tokens per second
                tokens_processed = batch_size * config["context_length"]
                tokens_per_second = tokens_processed / total_time_seconds

                # Compute FLOPs per token
                flops_per_token = total_flops / tokens_processed

                # Compute theoretical max tokens per second
                if flops_per_token > 0:
                    theoretical_max_tokens_per_second = max_flops_per_second / flops_per_token
                else:
                    theoretical_max_tokens_per_second = 0  # Avoid division by zero

                # Compute MFU
                if theoretical_max_tokens_per_second > 0:
                    mfu = tokens_per_second / theoretical_max_tokens_per_second
                else:
                    mfu = 0  # Avoid division by zero

                print(f"  Batch size {batch_size}: Tokens/sec: {tokens_per_second:.2f}, MFU: {mfu:.4f}")

                # If successful, try a larger batch size
                min_batch_size = batch_size + 1
                max_batch_size = batch_size

                # Clean up
                del model, input_tensor, output, loss
                torch.cuda.empty_cache()

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"  Batch size {batch_size}: Out of memory")
                    # Try smaller batch size
                    max_possible_batch_size = batch_size - 1

                    # Clean up
                    try:
                        del model, input_tensor
                        torch.cuda.empty_cache()
                    except NameError:
                        pass
                else:
                    raise e

else:
    print("Unknown GPU model. Please update the flops_per_second dictionary with your GPU information.")

**bfloat16**

In [ ]:
from torch._C import dtype
from supreme_robot import SupremeGPTModel

if gpu_model != "Unknown":

    for size in model_configs:
        print(f"\nProcessing {size}")
        config = BASE_CONFIG.copy()
        config.update(model_configs[size])

        min_batch_size = 1
        max_batch_size = None
        max_possible_batch_size = 4096

        while min_batch_size <= max_possible_batch_size:
            batch_size = (min_batch_size + max_possible_batch_size) // 2
            try:
                input_tensor = torch.randint(
                    0, config["vocab_size"],
                    (batch_size, config["context_length"]),
                    device=device
                )

                model = SupremeGPTModel(config).to(device, dtype=torch.float32)
                model.train()

                # Start timing
                torch.cuda.synchronize()
                start_time = time.time()

                # Forward & backward pass
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                  output = model(input_tensor)
                  loss = output.sum()  # Compute a dummy loss

                loss.backward()

                # End timing
                torch.cuda.synchronize()
                end_time = time.time()

                total_time_seconds = end_time - start_time

                # Calculate FLOPs for forward pass
                macs, params = profile(model, inputs=(input_tensor,), verbose=False)
                flops_forward = 2 * macs  # Assuming one MAC equals two FLOPs

                # Estimate FLOPs for backward pass (typically 2x forward FLOPs)
                flops_backward = 2 * flops_forward

                # Total FLOPs for forward + backward passes
                total_flops = flops_forward + flops_backward  # Or total_flops = flops_forward * 3

                data_type = next(model.parameters()).dtype
                max_flops_per_second = flops_per_second[gpu_model].get(data_type, 0)

                # Compute tokens per second
                tokens_processed = batch_size * config["context_length"]
                tokens_per_second = tokens_processed / total_time_seconds

                # Compute FLOPs per token
                flops_per_token = total_flops / tokens_processed

                # Compute theoretical max tokens per second
                if flops_per_token > 0:
                    theoretical_max_tokens_per_second = max_flops_per_second / flops_per_token
                else:
                    theoretical_max_tokens_per_second = 0  # Avoid division by zero

                # Compute MFU
                if theoretical_max_tokens_per_second > 0:
                    mfu = tokens_per_second / theoretical_max_tokens_per_second
                else:
                    mfu = 0  # Avoid division by zero

                print(f"  Batch size {batch_size}: Tokens/sec: {tokens_per_second:.2f}, MFU: {mfu:.4f}")

                # If successful, try a larger batch size
                min_batch_size = batch_size + 1
                max_batch_size = batch_size

                # Clean up
                del model, input_tensor, output, loss
                torch.cuda.empty_cache()

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"  Batch size {batch_size}: Out of memory")
                    # Try smaller batch size
                    max_possible_batch_size = batch_size - 1

                    # Clean up
                    try:
                        del model, input_tensor
                        torch.cuda.empty_cache()
                    except NameError:
                        pass
                else:
                    raise e

else:
    print("Unknown GPU model. Please update the flops_per_second dictionary with your GPU information.")

**All Torch**

In [2]:
import contextlib
import gc
import time

import torch
from torch.nn.attention import sdpa_kernel, SDPBackend
from torch.utils.flop_counter import FlopCounterMode

from supreme_robot_torch import SupremeGPTModel

FORCE_FLASH = True   # raise an error if the flash kernel can't be used (instead of silent fallback)
WARMUP_STEPS = 1     # untimed steps (CUDA init, kernel selection, allocator warmup)
TIMED_STEPS = 3      # timed steps, averaged
MAX_BATCH = 256


def attention_ctx():
    return sdpa_kernel(SDPBackend.FLASH_ATTENTION) if FORCE_FLASH else contextlib.nullcontext()


def train_step(model, input_tensor):
    with attention_ctx():
        output = model(input_tensor)
        loss = output.sum()  # dummy loss
        loss.backward()


def count_flops_per_token(model, config):
    """Forward + backward FLOPs per token, including attention inside SDPA (thop misses it)."""
    x = torch.randint(0, config["vocab_size"], (1, config["context_length"]), device=device)
    with FlopCounterMode(display=False) as fc:
        train_step(model, x)
    model.zero_grad(set_to_none=True)
    return fc.get_total_flops() / config["context_length"]


def time_train_step(model, config, batch_size):
    """Average seconds per forward+backward step. Locals are freed when this returns or raises."""
    input_tensor = torch.randint(
        0, config["vocab_size"],
        (batch_size, config["context_length"]),
        device=device
    )
    for _ in range(WARMUP_STEPS):
        train_step(model, input_tensor)
        model.zero_grad(set_to_none=True)

    torch.cuda.synchronize()
    start_time = time.time()
    for _ in range(TIMED_STEPS):
        train_step(model, input_tensor)
        model.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    return (time.time() - start_time) / TIMED_STEPS


def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
if gpu_model != "Unknown":

    for size in model_configs:
        print(f"\nProcessing {size}")
        config = BASE_CONFIG.copy()
        config.update(model_configs[size])

        # Build the model ONCE per size (pure bf16 weights -> Tensor Cores + FlashAttention)
        try:
            model = SupremeGPTModel(config).bfloat16().to(device)
             # Compile model
            #model = torch.compile(model)
            model.train()
            flops_per_token = count_flops_per_token(model, config)
        except torch.cuda.OutOfMemoryError:
            print("  Model does not fit in memory")
            cleanup()
            continue

        data_type = next(model.parameters()).dtype
        max_flops_per_second = flops_per_second[gpu_model].get(data_type, 0)
        if max_flops_per_second == 0:
            print(f"  Warning: no peak FLOPs for {data_type} in flops_per_second -> MFU will be 0")

        min_batch_size = 1
        max_batch_size = None
        max_possible_batch_size = 4096

        while min_batch_size <= max_possible_batch_size:
            batch_size = (min_batch_size + max_possible_batch_size) // 2
            oom = False
            try:
                seconds_per_step = time_train_step(model, config, batch_size)

                tokens_processed = batch_size * config["context_length"]
                tokens_per_second = tokens_processed / seconds_per_step

                # MFU = achieved FLOP/s / peak FLOP/s
                mfu = tokens_per_second * flops_per_token / max_flops_per_second if max_flops_per_second > 0 else 0

                print(f"  Batch size {batch_size}: Tokens/sec: {tokens_per_second:.2f}, MFU: {mfu:.4f}")

                # If successful, try a larger batch size
                min_batch_size = batch_size + 1
                max_batch_size = batch_size

            except torch.cuda.OutOfMemoryError:
                print(f"  Batch size {batch_size}: Out of memory")
                oom = True
                max_possible_batch_size = batch_size - 1

            # Cleanup happens after the except block so the traceback no longer holds tensors
            if oom:
                model.zero_grad(set_to_none=True)
            cleanup()

        print(f"  Max batch size that fit: {max_batch_size}")

        del model
        cleanup()

else:
    print("Unknown GPU model. Please update the flops_per_second dictionary with your GPU information.")

**Compiled**

In [ ]:
import torch._dynamo

torch._dynamo.config.cache_size_limit = 64

def time_step(run_model, config, batch_size):
    x = torch.randint(0, config["vocab_size"], (batch_size, config["context_length"]), device=device)
    for _ in range(WARMUP_STEPS):
        train_step(run_model, x)
        run_model.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(TIMED_STEPS):
        train_step(run_model, x)
        run_model.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    return (time.time() - start) / TIMED_STEPS

if gpu_model != "Unknown":
    for size, size_cfg in model_configs.items():
        print(f"\nProcessing {size}")
        config = {**BASE_CONFIG, **size_cfg}

        try:
            model = SupremeGPTModel(config).bfloat16().to(device)
            model.train()

            # FLOPs per token (fwd + bwd, incl. attention) from the eager model, batch size 1
            x1 = torch.randint(0, config["vocab_size"], (1, config["context_length"]), device=device)
            with FlopCounterMode(display=False) as fc:
                train_step(model, x1)
            model.zero_grad(set_to_none=True)
            flops_per_token = fc.get_total_flops() / config["context_length"]
            del x1
        except torch.cuda.OutOfMemoryError as e:
            print("  Model does not fit in memory:", str(e).split("\n")[0])
            cleanup()
            continue

        peak = flops_per_second[gpu_model].get(next(model.parameters()).dtype, 0)
        compiled_model = torch.compile(model, dynamic=False)  # shares parameters with `model`

        best, batch_size = None, 1
        while batch_size <= MAX_BATCH:
            oom = False
            try:
                seconds = time_step(compiled_model, config, batch_size)
                tps = batch_size * config["context_length"] / seconds
                mfu = tps * flops_per_token / peak if peak else 0.0
                print(f"  Batch size {batch_size}: Tokens/sec: {tps:.2f}, MFU: {mfu:.4f}")
                best = (batch_size, tps, mfu)
            except torch.cuda.OutOfMemoryError:
                print(f"  Batch size {batch_size}: Out of memory")
                oom = True
            if oom:
                model.zero_grad(set_to_none=True)
            cleanup()
            if oom:
                break
            batch_size *= 2

        if best:
            print(f"  -> best: batch {best[0]}, {best[1]:.0f} tokens/s, MFU {best[2]:.4f}")

        del compiled_model, model
        torch._dynamo.reset()
        cleanup()
else:
    print("Unknown GPU model. Please update the flops_per_second dictionary with your GPU information.")